In [0]:
from pyspark.sql import functions as F


In [0]:
date_cutoff = '2023-01-01'
material = 'ED13'
rent = spark.table('rentabilidad_material_silver_table').filter(
    (F.col('material') == 'ED13') &
    (F.to_date(F.col('week-month-year')) > F.lit(date_cutoff))
)
litros = spark.table('litros_material_silver_table')
metadata = spark.table('metadata_materiales_optimized')
ventas = spark.table('ventas_material_silver_table').filter(
    (F.col('material') == 'ED13') &
    (F.to_date(F.col('week-month-year')) > F.lit(date_cutoff))
)

In [0]:
cond = spark.table('condiciones_bronze_table').filter(
    (F.col('material') == 'ED13') &
    (F.col('pnd_code') == '50')
)

In [0]:
display(cond.limit(5))

In [0]:
display(rent.limit(5))
display(litros.limit(5))
display(metadata.limit(5))
display(ventas.limit(5))

In [0]:
merged_df = ventas.join(litros, ['week-month-year', 'establecimiento', 'material'], how='inner')

display(merged_df)

In [0]:
metadata = metadata.withColumn(
    "volumen",
    F.when(F.col("volumen").contains(" L"),
           F.regexp_replace(F.regexp_replace(F.col("volumen"), ",", "."), " L", "").cast("double"))
    .when(F.col("volumen").contains("DM3"),
           F.regexp_replace(F.regexp_replace(F.col("volumen"), ",", "."), " DM3", "").cast("double"))
    .when(F.col("volumen").contains("CM3"),  # ← check this BEFORE M3!
          F.regexp_replace(F.regexp_replace(F.col("volumen"), ",", "."), " CM3", "").cast("double") * 0.001)
    .when(F.col("volumen").contains("M3"),
          F.regexp_replace(F.regexp_replace(F.col("volumen"), ",", "."), " M3", "").cast("double") * 1000)
    .otherwise(None)
).select('material_id', 'volumen')
display(metadata)


In [0]:
merged_df = merged_df.join(
    metadata.select("material_id", "volumen"),
    merged_df.material == metadata.material_id,
    how="left"
)

merged_df = merged_df.withColumn(
    "n_material",
    F.round(F.try_divide(F.col("litros"), F.col("volumen")), 2)
)

merged_df = merged_df.withColumn(
    "euro_material",
    F.round(F.try_divide(F.col("ventas_netas"), F.col("n_material")), 2)
)
display(merged_df)

In [0]:
merged_df = rent.join(litros, ['week-month-year', 'establecimiento', 'material'], how='inner')
merged_df.count()

In [0]:
metadata = metadata.withColumn(
    "volumen",
    F.regexp_replace(F.regexp_replace(F.col("volumen"), ",", "."), " L", "").cast("double")
).select('material_id', 'volumen')

merged_df = merged_df.join(
    metadata.select("material_id", "volumen"),
    merged_df.material == metadata.material_id,
    how="left"
)

merged_df = merged_df.withColumn("euro_litro", F.try_divide(F.col("weekly_rentabilidad"), F.col("litros")))

merged_df = merged_df.withColumn(
    "n_material",
    F.round(F.try_divide(F.col("litros"), F.col("volumen")), 2)
)
merged_df = merged_df.withColumn("precio_material", F.col("volumen") * F.col("euro_litro"))

In [0]:
display(merged_df)

Por qué hay tanta variabilidad en el precio?

Seguro que la rentabilidad semanal está vinculada a los litros semanales?

Salen precios negativos.